Predictive Modeling script using Logistic Regression

In [ ]:
# Phase 4: Predictive Modeling - Logistic Regression
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Load dataset
workspace_root = Path.cwd()
data_path = workspace_root / "cleaned_ecommerce_returns.csv"
if not data_path.exists():
    data_path = workspace_root.parent / "cleaned_ecommerce_returns.csv"

df = pd.read_csv(data_path)

# --- 1. Feature Engineering ---
# Create price bands because they are not present in the dataset
df["price_band"] = pd.cut(
    df["price"],
    bins=[-float("inf"), 25, 100, 250, float("inf")],
    labels=["Low", "Medium", "High", "Premium"],
)

# Fill missing values before encoding
for col in df.columns:
    if isinstance(df[col].dtype, pd.CategoricalDtype):
        df[col] = df[col].astype(str).fillna("Missing")
    elif df[col].dtype == "object":
        df[col] = df[col].fillna("Missing")
    else:
        df[col] = df[col].fillna(0)

# Encode categorical variables that exist in the dataset
categorical_cols = ["category", "region", "price_band"]
if "brand" in df.columns:
    categorical_cols.append("brand")

for col in categorical_cols:
    df[col] = df[col].astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Features (X) and target (y)
feature_cols = ["category", "price", "region", "price_band"]
if "brand" in df.columns:
    feature_cols.insert(1, "brand")

X = df[feature_cols]
y = df["is_return"]

# --- 2. Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --- 3. Scaling ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4. Logistic Regression Model ---
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# --- 5. Predictions ---
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# --- 6. Evaluation ---
print(" Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n Classification Report:\n", classification_report(y_test, y_pred))
print("\n ROC-AUC Score:", roc_auc_score(y_test, y_prob))

# --- 7. Example: Predict probability of return for first 5 test rows ---
sample_indices = X_test.index[:5]
sample_probs = pd.DataFrame({
    "order_id": df.loc[sample_indices, "order_id"].values,
    "return_probability": y_prob[:5],
})
print("\n🔮 Sample Predictions:\n", sample_probs)


📊 Confusion Matrix:
 [[9779    0]
 [ 571    0]]

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.94      1.00      0.97      9779
           1       0.00      0.00      0.00       571

    accuracy                           0.94     10350
   macro avg       0.47      0.50      0.49     10350
weighted avg       0.89      0.94      0.92     10350


📊 ROC-AUC Score: 0.5515353408399177


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


ValueError: All arrays must be of the same length